In [0]:
dbutils.widgets.text("input_path", "")
dbutils.widgets.text("output_path", "")

input_path = dbutils.widgets.get("input_path").strip()
output_path = dbutils.widgets.get("output_path").strip()

In [0]:
if not input_path or not output_path:
    raise ValueError(
        "Both input_path and output_path must be set. "
        "Example input: /Volumes/dev_automotive/landing/landing_raw/final_geografic.csv | "
        "Example output: /Volumes/dev_automotive/landing/landing_raw/final_geografic.xml"
    )

print(f"Input (CSV):  {input_path}")
print(f"Output (XML): {output_path}")

In [0]:
import csv
import xml.etree.ElementTree as ET
from datetime import datetime

def csv_to_xml(csv_file, xml_file):
    """Read CSV into list of dicts, add audit columns, write single XML file (UTF-8, indented)."""
    try:
        with open(csv_file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            data = list(reader)

        if not data:
            print("No data in the file")
            return 0

        # Audit columns (Databricks best practice)
        loaded_at = datetime.utcnow().isoformat() + "Z"
        for row in data:
            row["loaded_at"] = loaded_at
            row["source_file"] = csv_file

        root = ET.Element("data")
        for row in data:
            record = ET.SubElement(root, "record")
            for key, value in row.items():
                elem = ET.SubElement(record, key)
                elem.text = str(value) if value is not None else ""

        ET.indent(root, space="  ")
        tree = ET.ElementTree(root)
        tree.write(xml_file, encoding="utf-8", xml_declaration=True, default_namespace=None)

        print(f"Successfully converted {csv_file} to {xml_file}")
        return len(data)

    except FileNotFoundError:
        print(f"Error: File {csv_file} not found")
        raise
    except Exception as e:
        print(f"Error: {e}")
        raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Run conversion and verify count

# COMMAND ----------

record_count = csv_to_xml(input_path, output_path)
print(f"Records written to XML: {record_count}")
